In [20]:
import pandas as pd 
from pandas import CategoricalDtype 
import numpy as np
import sqlite3
import plotnine as pm
import scipy.stats as stats
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')


In [21]:
intake_cleaned = pd.read_csv('/kaggle/input/datasets/micahluftig/cleaned-data-from-file/intake_sorted.csv')
outcome_cleaned = pd.read_csv('/kaggle/input/datasets/micahluftig/cleaned-data-from-file/outcome_sorted.csv')
weather_sorted = pd.read_csv('/kaggle/input/datasets/micahluftig/cleaned-data-from-file/weather_sorted.csv')

In [22]:
conn = sqlite3.connect('shelter_data.db')

#conn.execute('DROP TABLE IF EXISTS intake')
#conn.execute('DROP TABLE IF EXISTS outcome')

# Check if the 'intake' table already exists
try:
    # Create tables if they do not already exist
    apt_data.to_sql('apt_data', conn, index=False, if_exists='replace')
    p_data.to_sql('p_data', conn, index=False, if_exists='replace')
    

    print('Database tables created successfully.')

except ValueError:
    print('Tables already exist. Skipping creation step.')

#first im going to create a table that has the k9 appointment data
query = """
SELECT  
    i.appointment_id AS intake_id,
    i.animal_id,
    i.datetime AS intake_time, 
    MIN(o.datetime) AS discharge_time, 
    i.intake_type,
    i.intake_reason, 
    o.outcome_category,
    o.outcome_subcategory,
    o.shift,
    ROUND(julianday(MIN(o.datetime)) - julianday(i.datetime), 2) AS apt_duration_days

FROM intake AS i

LEFT JOIN outcome AS o ON 
    i.animal_id = o.animal_id AND 
    o.datetime > i.datetime

GROUP BY i.animal_id, i.datetime

ORDER BY i.datetime;
"""
apt_data = pd.read_sql_query(query, conn)



print('*!*!' * 30)
print('apt data:')
print(apt_data.head())
print('\n\n\n')
query = """
SELECT  
    i.animal_id,
    i.cln_name,
    i.name_given_at_intake,
    i.spp,
    o.age_out,
    o.sex_out,
    i.pregnant_o_nursing
    
FROM intake AS i

LEFT JOIN outcome AS o ON 
    i.animal_id = o.animal_id

WHERE i.datetime = (
    SELECT MAX(datetime) 
    FROM intake 
    WHERE animal_id = i.animal_id
    )    

GROUP BY i.animal_id

ORDER BY i.animal_id;
"""

p_data = pd.read_sql_query(query, conn)
#p_data.to_csv('p_data.csv', index = False)
print('*!*!' * 30)
print('p data:')
print(p_data.head())
print('\n\n\n')
print('*!*!' * 30)



Database tables created successfully.
*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!*!
apt data:
                     intake_id animal_id          intake_time  \
0  a521520_2013-10-01 07:51:00   a521520  2013-10-01 07:51:00   
1  a664235_2013-10-01 08:33:00   a664235  2013-10-01 08:33:00   
2  a664236_2013-10-01 08:33:00   a664236  2013-10-01 08:33:00   
3  a664237_2013-10-01 08:33:00   a664237  2013-10-01 08:33:00   
4  a664233_2013-10-01 08:53:00   a664233  2013-10-01 08:53:00   

        discharge_time intake_type intake_reason outcome_category  \
0  2013-10-01 15:39:00       stray       routine            alive   
1  2013-10-01 10:39:00       stray       routine            admin   
2  2013-10-01 10:44:00       stray       routine            admin   
3  2013-10-01 10:44:00       stray       routine            admin   
4  2013-10-01 15:33:00       stray       medical         deceased   

  outcome_subcategory shi

In [23]:

query = """
WITH no_dupe_moon AS (
    SELECT DISTINCT 
        strftime('%Y', datetime) AS year,
        DATE(datetime) AS date_only, 
        cln_moonphase 
    FROM weather
),

appts_per_year_moon AS (
    SELECT 
        m.year,
        m.cln_moonphase AS moonphase, 
        COUNT(*) AS total_apts
    FROM intake AS i
    JOIN no_dupe_moon AS m ON DATE(i.datetime) = m.date_only
    GROUP BY m.year, m.cln_moonphase
),

deceased_per_year_moon AS (
    SELECT 
        m.year,
        m.cln_moonphase AS moonphase, 
        COUNT(*) AS p_deceased
    FROM apt AS a
    JOIN no_dupe_moon AS m ON DATE(a.discharge_time) = m.date_only
    WHERE
        a.outcome_category = 'deceased'
    GROUP BY m.year, m.cln_moonphase
),

days_per_year_moon AS (
    SELECT 
        year,
        cln_moonphase AS moonphase, 
        COUNT(*) AS num_days
    FROM no_dupe_moon
    GROUP BY year, moonphase
)


SELECT 
    a.year,
    a.moonphase,
    (CAST(a.total_apts AS FLOAT) / day.num_days) AS avg_daily_intakes,
    (CAST(dec.p_deceased AS FLOAT) / day.num_days) AS avg_p_deceased
FROM appts_per_year_moon a
JOIN days_per_year_moon day ON a.year = day.year AND a.moonphase = day.moonphase
JOIN deceased_per_year_moon dec ON dec.year = day.year AND dec.moonphase = day.moonphase
ORDER BY a.year ASC, avg_daily_intakes DESC;

"""
query = pd.read_sql_query(query, conn)

print(query)

query.to_csv('avg_appt_per_moonphase.csv', index = False)


Database tables created successfully.
    year        moonphase  avg_daily_intakes  avg_p_deceased
0   2013   waxing_gibbous          11.815789        0.842105
1   2013             full          11.083333        0.333333
2   2013    first_quarter          10.916667        0.416667
3   2013         new_moon          10.916667        0.666667
4   2013  waning_crescent          10.870588        0.823529
..   ...              ...                ...             ...
64  2021   waxing_gibbous           3.567568        0.067568
65  2021   waning_gibbous           3.000000        0.111111
66  2021    third_quarter           2.642857        0.071429
67  2021  waning_crescent           1.939024        0.060976
68  2021  waxing_crescent           1.684211        0.052632

[69 rows x 4 columns]


In [25]:



query = """ 
WITH no_dupe_weather AS ( 
    SELECT DISTINCT 
        DATE(datetime) AS date, 
        "temp(°F)", 
        "feels_like(°F)", 
        "humidity(%)", 
        "precip(in)", 
        "snow(in)", 
        "snow_depth(in)", 
        "wind_gust(mph)", 
        "wind_speed(mph)", 
        "sea_level_pressure(mb)", 
        "visibility(mi)", 
        "solar_radiation(W/m²)", 
        "solar_energy(MJ/m²)", 
        "uv_index" 
    FROM weather 
) 
SELECT 
    w.*, 
    COUNT(a.intake_time) AS apt_seen, 
    COUNT(a.outcome_category) AS p_deceased 
FROM no_dupe_weather AS w 
JOIN apt AS a ON DATE(a.intake_time) = w.date 
GROUP BY w.date; 
"""
weather_y_apts_per_day = pd.read_sql_query(query, conn)

print(weather_y_apts_per_day)

weather_y_apts_per_day.to_csv('other_environmental_variables.csv', index = False)
print(weather_y_apts_per_day.dtypes)
conn.close()


            date  temp(°F)  feels_like(°F)  humidity(%)  precip(in)  snow(in)  \
0     2013-10-01      77.8            79.1         72.3       0.000       0.0   
1     2013-10-02      81.0            83.5         73.5       0.000       0.0   
2     2013-10-03      82.1            84.0         69.5       0.000       0.0   
3     2013-10-04      80.6            82.5         71.3       0.000       0.0   
4     2013-10-05      76.1            76.6         67.7       0.000       0.0   
...          ...       ...             ...          ...         ...       ...   
2703  2021-02-27      66.8            66.8         79.7       0.009       0.0   
2704  2021-02-28      71.1            71.1         77.3       0.000       0.0   
2705  2021-03-01      53.1            52.9         64.9       0.106       0.0   
2706  2021-03-02      54.4            53.8         49.7       0.000       0.0   
2707  2021-03-03      55.6            55.6         52.9       0.000       0.0   

      snow_depth(in)  wind_

In [ ]:
p_data.to_csv('p_data.csv', index = False)
apt_data.to_csv('apt_data.csv', index = False)
query.to_csv('avg_appt_per_moonphase.csv', index = False)